# 02. Walk-forward, purge/embargo и as-of

Ноутбук показывает, почему случайный split и «подглядывание» будущего завышают качество. Данные синтетические; ClickHouse не нужен.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
months = pd.period_range("2018-01", periods=96, freq="M")
trend = 0.15 * np.arange(len(months))
season = 2.0 * np.sin(2 * np.pi * np.arange(len(months)) / 12)
y = trend + season + rng.normal(0, 0.6, len(months))
series = pd.Series(y, index=months, name="y")
series.head()

In [ ]:
def walk_forward(series, start=36, embargo=1):
    """Фолды: train — только прошлое; embargo вырезает месяцы перед тестом."""
    folds = []
    for t in range(start, len(series)):
        train_end = t - embargo
        if train_end <= 0:
            continue
        folds.append((series.iloc[:train_end], float(series.iloc[t]), t))
    return folds


def forecast_last3(train):
    return float(train.iloc[-3:].mean())


folds = walk_forward(series, start=36, embargo=1)
errors = [forecast_last3(train) - actual for train, actual, _ in folds]
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print(f"folds={len(folds)}, RMSE (last-3 mean) = {rmse(errors):.3f}")

In [ ]:
# Нарушение as-of: признак использует сам тестовый месяц (центрированное окно).
leaky = series.rolling(3, center=True, min_periods=1).mean()
leak_errors = [float(leaky.iloc[t]) - float(series.iloc[t]) for _, _, t in folds]
print(f"RMSE с утечкой (центрированное окно) = {rmse(leak_errors):.3f}")
print("Разница — это и есть цена подглядывания в будущее, а не улучшение модели.")

## Правила

- Случайный `train_test_split` для временных рядов запрещён.
- `embargo`/purge вырезает месяцы между train и test, чтобы лаги и таргет не пересекались.
- Признак, опубликованный после cutoff, в фолд не попадает: as-of проверяется тестом, а не глазами.
